# 04 · Stability
Rémi · HEC Match

Does changing the training sessions change the recommendation?

We keep the group’s train/test split and v0 hyperparameters. Whole sessions are resampled, keeping participants and reciprocal encounters together. Logit and XGBoost are refitted. TabICL uses frozen predictions for test uncertainty only; its training stability is not measured.

To regenerate the analysis: `python -m src.stability --refits 200 --eval-bootstrap 2000`. This notebook reads the exported results; opening it does not trigger hundreds of fits.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = Path.cwd()
if not (ROOT / "data/features.json").exists():
    raise RuntimeError("Run this notebook from the repository root.")
OUT = ROOT / "reports/stability"
s = json.loads((OUT / "summary.json").read_text())
if "features_by_model" not in s or "tabicl" not in s:
    raise RuntimeError("Historical results: rerun stability with the current feature contract before using this notebook.")
r = pd.read_csv(OUT / "refits.csv")
by_session = pd.read_csv(OUT / "by_session.csv")
# Refuse to silently display results generated from a different data contract.
import hashlib
for name, expected in s["inputs_sha256"].items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest() == expected, f"Stale results: {name}"
print(f"{s['train_rows']:,} training rows in {len(s['train_waves'])} sessions; "
      f"{s['test_rows']:,} test rows in {len(s['test_waves'])} sessions.")
print(f"{s['n_refits']} training-session resamples; {s['n_eval']} test-session resamples.")
print("Features per model:", {name: len(cols) for name, cols in s["features_by_model"].items()})
print("Logit contract:", s["logit_feature_source"])
print("TabICL:", s["tabicl"]["status"])
plt.rcParams.update({"font.family":"DejaVu Sans", "axes.spines.top":False,
                     "axes.spines.right":False, "figure.facecolor":"#f8f6f0",
                     "axes.facecolor":"#f8f6f0", "axes.titleweight":"bold"})
colors = {"logit":"#215544", "xgb":"#b3863c", "tabicl":"#536c9c"}

## Results
The threshold is fixed at 0.5 for this analysis. It is not an optimized business rule. A decision flip means that a refitted model crosses that threshold relative to its reference model, on the same test encounter.

In [ ]:
table = []
for name, m in s["models"].items():
    ci=m["test_auc_interval"]
    table.append({"Model":name, "Reference AUC":m["reference_auc"],
                  "Test AUC 2.5%":ci["p025"], "Test AUC 97.5%":ci["p975"],
                  "Mean flips (%)":100*m["flip_rate"]["mean"] if m["training_stability_available"] else np.nan,
                  "Mean probability change":m["probability_mae"]["mean"] if m["training_stability_available"] else np.nan})
display(pd.DataFrame(table).set_index("Model").round(3))
d=s["paired_auc_difference"]
print(f"Paired logit minus XGBoost AUC: [{d['p025']:.3f}, {d['p975']:.3f}].")
print("Only six test sessions: treat these intervals as exploratory.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,4), layout="constrained")
for name in colors:
    part=by_session[by_session.model==name]
    axes[0].plot(part.wave.astype(str), part.auc, marker="o", label=name, color=colors[name])
    if name in s["models_refitted"]:
        axes[1].hist(100*r.loc[r.model==name,"flip_rate"], bins=15, alpha=.6, label=name, color=colors[name])
axes[0].set(title="Performance by held-out session", xlabel="Session", ylabel="ROC AUC", ylim=(.4,.8))
axes[0].axhline(.5, color="#aaa", lw=1)
axes[1].set(title="Sensitivity to training sessions", xlabel="Decisions that change (%)", ylabel="Resamples")
for ax in axes: ax.legend(frameon=False)
fig.savefig(OUT/"stability.png", dpi=160)
plt.close(fig)
display(Image(filename=str(OUT/"stability.png")))

## Coefficient stability
Logit coefficients are converted to a common unit: one standard deviation in the original training set. This prevents a new scaler in each resample from creating artificial coefficient changes. The bars show the central 95% of refitted coefficients, not classical coefficient confidence intervals.

XGBoost’s gain-vector cosine distance is exported separately in `refits.csv`. Compare this distance within XGBoost, not against the logit coefficient distance.

In [ ]:
coef=pd.read_csv(OUT/"logit_coefficients.csv")
top=coef.loc[coef.reference_beta.abs().nlargest(12).index].sort_values("reference_beta")
fig,ax=plt.subplots(figsize=(9,5), layout="constrained")
y=np.arange(len(top))
ax.hlines(y, top.p025, top.p975, color="#9cae9e", lw=3)
ax.scatter(top.reference_beta, y, color=colors["logit"], zorder=3)
ax.axvline(0, color="#aaa", lw=1)
ax.set(yticks=y, yticklabels=top.feature, xlabel="Logit coefficient per training-set SD",
       title="Largest reference coefficients")
fig.savefig(OUT/"coefficients.png", dpi=160)
plt.close(fig)
display(Image(filename=str(OUT/"coefficients.png")))
display(top[["feature","reference_beta","p025","p975","sign_agreement"]].round(3))

In [ ]:
# Distances are comparable within a model, not across coefficient/gain units.
display(r.groupby("model")[["vector_euclidean_distance", "vector_cosine_distance"]].agg(["mean", "median"]).round(3))
display(pd.DataFrame(s["paired_auc_differences"]).T.round(3))
print("TabICL: no refits, no flip rates, no coefficient/importance distance.")

## Before the final presentation
- Integrate final logit/XGBoost configurations. TabICL test-session resamples use frozen predictions only; training sensitivity requires a separate GPU experiment.
- Recompute after feature selection and the business threshold are fixed. Keep the test set out of tuning.
- Discuss the trade-off between performance and changed recommendations, together with Blanche’s fairness audit and Max’s explanations.

These data describe decisions after short encounters, not long-term compatibility. The study does not establish that a dating app should use this model in production.

For Oli: `reports/stability/summary.json`, `by_session.csv` and the two PNG charts can feed the stability tab. All exported tables are aggregate; no participant-level predictions are published.